# 0. Submission information

**Submission date:** 25 November 2026

**QPM 2026-2027: Assignment 3**

**Group members:** LOLLI Alessandro, MASTROGIACOMO Paolo, PERALDI Léandre

**Comments/challenges:** Prices are downloaded from Yahoo Finance with `yfinance` (adjusted for splits and dividends) and the monthly simple excess returns run from January 2015 to December 2022 (96 months; risk-free rate = 0). The market weights are the S&P 500 index weights reported by Investopedia, rescaled to sum to one. The main choice in the Black–Litterman model is the uncertainty of the views: we set $\tau = 1/T$ and $\Omega = \mathrm{diag}(P\,\tau\Sigma\,P^\top)$ (He and Litterman, 1999), and we explain this choice in Q1.4. All quantities are monthly unless we say "annualised". We used Claude (Anthropic) to help structure and debug the code; all results and interpretations were checked by the group.

# 1. Question 1

*In this question, we determine the optimal portfolio weights for an investor who is considering investing in an asset universe that consists of only six stocks: AAPL, MSFT, AMZN, NVDA, TSLA, and META.*

*Please download prices for these six stocks and compute their monthly simple (not log) excess returns starting January 2015 and ending December 2022 (so you need the month-end price of December 2014), assuming that the risk-free rate is 0.*

*Use the "Index Weighting" reported in this article from Investopedia to determine the market weights for these assets (you may also be able to get the market capitalizations from Yahoo Finance).*

### Setup and data

We download daily adjusted prices from 1 December 2014 to 31 December 2022, keep the last price of each month, and compute simple returns. The first return is January 2015 (it uses the month-end price of December 2014) and the last is December 2022, so we have $T = 96$ monthly observations. Because the risk-free rate is 0, these returns are also excess returns.

In [ ]:
# !pip install -q yfinance        # uncomment if yfinance is not installed (e.g. in Colab)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from IPython.display import display, Markdown

pd.set_option("display.precision", 4)
pd.set_option("display.width", 120)

tickers = ["AAPL", "MSFT", "AMZN", "NVDA", "TSLA", "META"]

# Daily prices adjusted for splits and dividends
daily = yf.download(tickers, start="2014-12-01", end="2023-01-01",
                    auto_adjust=True, progress=False)["Close"][tickers]

# Month-end prices (December 2014 to December 2022) and simple monthly returns
prices = daily.resample("ME").last().loc["2014-12":"2022-12"]
R = prices.pct_change().dropna()          # excess returns, since Rf = 0

assert len(R) == 96 and R.notna().all().all(), "Check the data download"
print(f"Monthly returns from {R.index[0]:%B %Y} to {R.index[-1]:%B %Y}: T = {len(R)}")
R.head()

In [ ]:
# Sample moments (monthly), used throughout the question
T = len(R)
mu = R.mean()                 # sample mean excess returns
Sigma = R.cov()               # sample covariance matrix (divides by T-1)

stats = pd.DataFrame({
    "mean (monthly)": mu,
    "std (monthly)": R.std(),
    "mean (annual)": 12 * mu,
    "std (annual)": np.sqrt(12) * R.std(),
    "Sharpe (annual)": np.sqrt(12) * mu / R.std(),
    "s.e. of mean (annual)": 12 * R.std() / np.sqrt(T),
})
display(stats.round(4))
print("Correlation matrix:")
display(R.corr().round(2))

## Q1.1

*Based on the data on index weighting (or market capitalization) you have downloaded, write down the "market" weights of a portfolio that invests in only AAPL, MSFT, AMZN, NVDA, TSLA, and META. Rescale the weights so that they sum to one, because these six stocks are the whole market in this question. Then compute the market's risk aversion implied by the data, $\gamma_{mkt} = E[R_{mkt}]/V[R_{mkt}]$, where $R_{mkt}$ is the excess return on this market portfolio. Use $\gamma_{mkt}$ throughout the rest of this question.*

### Answer

We take the "Index Weighting" (percentage of the S&P 500) of each stock from the Investopedia article *Top 10 S&P 500 Stocks by Index Weight* and divide each weight by the sum of the six weights, so that the six stocks form the whole market:
$$
w_{mkt,i} = \frac{\text{index weight}_i}{\sum_{j=1}^{6}\text{index weight}_j}.
$$
The market excess return is $R_{mkt,t} = w_{mkt}^\top R_t$, and the implied risk aversion is $\gamma_{mkt} = E[R_{mkt}]/V[R_{mkt}]$, computed with the sample mean and variance of the 96 monthly returns. Since $V[R_{mkt}] = w_{mkt}^\top \Sigma\, w_{mkt}$, $\gamma_{mkt}$ is the risk aversion that makes the market portfolio the mean–variance optimal portfolio (Lecture 5).

In [ ]:
# S&P 500 index weights (%) from Investopedia, "Top 10 S&P 500 Stocks by Index Weight"
# https://www.investopedia.com/top-10-s-and-p-500-stocks-by-index-weight-4843111
index_weight = pd.Series({
    "AAPL": 6.19,
    "MSFT": 4.97,
    "AMZN": 3.75,
    "NVDA": 7.08,
    "TSLA": 2.44,
    "META": 2.67,
})[tickers]

w_mkt = index_weight / index_weight.sum()      # rescaled: sums to one

R_mkt = R @ w_mkt                               # market excess return
gamma_mkt = R_mkt.mean() / R_mkt.var()          # E[R_mkt] / V[R_mkt]

display(pd.DataFrame({"index weight (%)": index_weight,
                      "market weight w_mkt": w_mkt}).round(4))
print(f"Sum of market weights: {w_mkt.sum():.4f}")
print(f"Market: mean {R_mkt.mean():.4%} per month, variance {R_mkt.var():.6f}, "
      f"volatility {R_mkt.std() * np.sqrt(12):.2%} per year")
print(f"Implied risk aversion gamma_mkt = {gamma_mkt:.4f}")

In [ ]:
display(Markdown(
    f"**Result.** The rescaled market weights are "
    + ", ".join(f"{t} {w_mkt[t]:.1%}" for t in tickers)
    + f". The market portfolio of these six stocks earned an average excess return of "
    f"{R_mkt.mean():.2%} per month with a variance of {R_mkt.var():.5f} "
    f"(volatility {R_mkt.std() * np.sqrt(12):.1%} per year), so "
    f"$\\gamma_{{mkt}} = {R_mkt.mean():.5f}/{R_mkt.var():.5f} = {gamma_mkt:.3f}$. "
    f"This is the risk aversion of an investor for whom the market portfolio is optimal; "
    f"we use it in all the following parts."
))

## Q1.2

*Based on the sample data on returns, compute the Markowitz mean-variance portfolio weights for $\gamma = \gamma_{mkt}$. Are these weights reasonable? Explain why.*

### Answer

With a risk-free asset (rate 0), the Markowitz portfolio of risky assets is
$$
w_{MV} = \frac{1}{\gamma_{mkt}}\,\hat\Sigma^{-1}\hat\mu ,
$$
where $\hat\mu$ and $\hat\Sigma$ are the sample mean and covariance of monthly excess returns. The weights need not sum to one: $1 - \mathbf 1^\top w_{MV}$ is invested in the risk-free asset (a negative value means borrowing).

In [ ]:
w_mv = pd.Series(np.linalg.solve(Sigma, mu) / gamma_mkt, index=tickers)

out = pd.DataFrame({"Markowitz w_MV": w_mv, "market w_mkt": w_mkt})
display(out.round(4))
print(f"Sum of risky weights: {w_mv.sum():.4f}   ->   risk-free weight: {1 - w_mv.sum():.4f}")
print(f"Gross exposure sum|w|: {w_mv.abs().sum():.4f}")

In [ ]:
longs = w_mv[w_mv > 0].sort_values(ascending=False)
shorts = w_mv[w_mv < 0].sort_values()
txt = (f"**Result.** The Markowitz weights sum to {w_mv.sum():.2f} "
       f"(so {1 - w_mv.sum():.2f} is in the risk-free asset"
       + (", i.e. the investor borrows" if w_mv.sum() > 1 else "") + "), "
       f"and the gross exposure is {w_mv.abs().sum():.2f}. "
       f"The largest long position is {longs.index[0]} ({longs.iloc[0]:.1%}). ")
if len(shorts) > 0:
    txt += ("The portfolio is short " + ", ".join(f"{t} ({shorts[t]:.1%})" for t in shorts.index) + ". ")
else:
    txt += "There are no short positions, but the weights are very different from the market weights. "
txt += ("By comparison, the market weights are all positive, sum to one, and the largest is "
        f"{w_mkt.max():.1%}.")
display(Markdown(txt))

**Are these weights reasonable? No.**

1. **They are extreme.** The weights shown above are far from the market weights: they contain large long and short positions, and the total exposure to risky assets (and the amount borrowed or lent) is very different from what an investor with the *market's* risk aversion should hold. By construction, an investor with risk aversion $\gamma_{mkt}$ should hold exactly the market portfolio if the market is mean–variance efficient (this is what Q1.3 uses). The Markowitz weights are very different because they use the *sample* means.
2. **They are driven by estimation error in the means.** The table of sample moments shows that, with only 96 months, the standard error of each annual mean return is large: roughly 7–20 percentage points per year (largest for NVDA and TSLA). The optimiser treats these noisy sample means as the truth. It takes large long positions in the stocks with the highest *past* means (e.g. NVDA, which had an exceptional 2015–2022) and short positions, or small positions, in the others.
3. **They are amplified by the high correlations.** The six stocks are all large technology-related firms and are strongly positively correlated. $\hat\Sigma^{-1}$ then turns small differences in means into large long–short bets between similar stocks. This is the "error maximisation" problem of Michaud (1989) discussed in Lecture 4.
4. **They are unstable and perform badly out of sample.** Small changes in $\hat\mu$ (e.g. a different sample period) would change these weights a lot. As we saw in Lectures 2–4, such portfolios typically have poor out-of-sample performance.

These problems are exactly the motivation for the Black–Litterman model.

*Next, we will compute the Black-Litterman portfolio weights. We will do this in four small steps.*

## Q1.3

*Step 1: Using the rescaled market weights and $\gamma_{mkt}$ from Q1.1, calculate the CAPM-implied expected returns.*

### Answer

If the market portfolio is the mean–variance portfolio of an investor with risk aversion $\gamma_{mkt}$, then $w_{mkt} = \frac{1}{\gamma_{mkt}}\Sigma^{-1}\mu_{CAPM}$. Inverting this relation (*reverse optimisation*) gives the CAPM-implied (equilibrium) expected excess returns
$$
\mu_{CAPM} = \gamma_{mkt}\,\Sigma\, w_{mkt}.
$$
We use the sample covariance matrix $\hat\Sigma$, which is estimated much more precisely than the means.

In [ ]:
mu_capm = gamma_mkt * Sigma @ w_mkt

display(pd.DataFrame({
    "mu_CAPM (monthly)": mu_capm,
    "mu_CAPM (annual)": 12 * mu_capm,
    "sample mean (annual)": 12 * mu,
    "beta w.r.t. market": (Sigma @ w_mkt) / (w_mkt @ Sigma @ w_mkt),
}).round(4))

# Check: Markowitz with the CAPM-implied means gives back the market weights
w_check = np.linalg.solve(Sigma, mu_capm) / gamma_mkt
print("Markowitz weights using mu_CAPM:", np.round(w_check, 4))
print("Equal to w_mkt:", np.allclose(w_check, w_mkt))
print(f"Check: w_mkt' mu_CAPM = {w_mkt @ mu_capm:.5f} = E[R_mkt] = {R_mkt.mean():.5f}")

In [ ]:
display(Markdown(
    "**Result.** The CAPM-implied expected excess returns (annualised) are "
    + ", ".join(f"{t} {12 * mu_capm[t]:.1%}" for t in tickers)
    + f". They range from {12 * mu_capm.min():.1%} to {12 * mu_capm.max():.1%} per year, "
    f"a much narrower range than the sample means "
    f"({12 * mu.min():.1%} to {12 * mu.max():.1%}). "
    "Each stock's CAPM return is proportional to its covariance with the market, "
    "$\\mu_{CAPM,i} = \\beta_i\\,E[R_{mkt}]$, so high-beta stocks (e.g. TSLA, NVDA) have higher expected returns. "
    "As a check, using $\\mu_{CAPM}$ in the Markowitz formula gives back exactly the market weights, "
    "and the market's CAPM expected return equals its sample mean."
))

## Q1.4

*Step 2: Specify the pick matrix $P$ and the view vector $q$ that captures the following views for each of the assets:*
- *AAPL: its absolute excess return is expected to be 10% per year.*
- *MSFT: its absolute excess return is expected to be 5% per year.*
- *AMZN: no views*
- *NVDA will outperform TSLA by 2% per year.*
- *TSLA will underperform META by 1% per year (that is, META − TSLA = 1%).*

*Convert the views to monthly terms before you use them, because the returns are monthly.*
*Finally, explain your choice for the matrix $\Omega$, which captures the uncertainty about these views. (There are different ways to specify this matrix, so feel free to choose anything reasonable.)*

### Answer

**Views.** With the assets ordered as (AAPL, MSFT, AMZN, NVDA, TSLA, META), we have $K = 4$ views, $P\mu = q + \varepsilon_q$ with $\varepsilon_q \sim N(0, \Omega)$:
$$
P = \begin{pmatrix}
1 & 0 & 0 & 0 & 0 & 0\\
0 & 1 & 0 & 0 & 0 & 0\\
0 & 0 & 0 & 1 & -1 & 0\\
0 & 0 & 0 & 0 & -1 & 1
\end{pmatrix},
\qquad
q = \frac{1}{12}\begin{pmatrix}0.10\\0.05\\0.02\\0.01\end{pmatrix}
= \begin{pmatrix}0.00833\\0.00417\\0.00167\\0.00083\end{pmatrix}.
$$
- Rows 1 and 2 are **absolute** views (a single 1).
- Rows 3 and 4 are **relative** views: each row sums to zero (a long–short portfolio). Row 4 is written as META − TSLA = 1%, as in the question.
- **AMZN has no view**, so its column of $P$ is all zeros.
- We convert the annual views to monthly by dividing by 12, consistent with the monthly means.

**Choice of $\Omega$.** We follow He and Litterman (1999), as in Lecture 5:
$$
V_\mu = \tau\Sigma,\quad \tau = \frac{1}{T} = \frac{1}{96},\qquad
\Omega = \mathrm{diag}\big(P\,(\tau\Sigma)\,P^\top\big).
$$
- $\tau = 1/T$: the uncertainty about the mean, $\tau\Sigma$, is the variance of a sample mean estimated from $T = 96$ months.
- The variance of each view equals the variance of the *portfolio the view is about* (row $k$ of $P$), scaled by $\tau$. Views on volatile stocks or volatile long–short portfolios are therefore less certain. For example, the view on NVDA − TSLA, two very volatile stocks, is less precise than the view on MSFT.
- With this choice, the investor is as confident in each view as in the CAPM prior for that portfolio. For a single view, the posterior would put a weight of exactly 50% on the view and 50% on the CAPM.
- $\Omega$ is diagonal: the errors in different views are independent.
- With $\Omega \propto \tau$, the value of $\tau$ cancels out of $\mu_{BL}$ (Lecture 5), so the posterior means do not depend on the exact choice of $\tau$.

Other reasonable choices would be the Idzorek (2007) method, which specifies a confidence level in % for each view, or a scalar multiple of the He–Litterman $\Omega$ to make the investor more or less confident.

In [ ]:
assets = tickers                                    # AAPL, MSFT, AMZN, NVDA, TSLA, META
views = ["AAPL = 10%", "MSFT = 5%", "NVDA - TSLA = 2%", "META - TSLA = 1%"]

P = pd.DataFrame([[1, 0, 0, 0,  0, 0],
                  [0, 1, 0, 0,  0, 0],
                  [0, 0, 0, 1, -1, 0],
                  [0, 0, 0, 0, -1, 1]], index=views, columns=assets, dtype=float)
q_annual = pd.Series([0.10, 0.05, 0.02, 0.01], index=views)
q = q_annual / 12                                   # monthly views

tau = 1 / T
Omega = pd.DataFrame(np.diag(np.diag(P @ (tau * Sigma) @ P.T)), index=views, columns=views)

print("Pick matrix P:"); display(P)
print("Views q (monthly):"); display(q.round(5))
print(f"tau = 1/T = {tau:.5f}")
print("Omega = diag(P tau Sigma P'):"); display(Omega.round(6))

In [ ]:
# How different are the views from what the CAPM already implies?
cmp = pd.DataFrame({
    "view q (annual)": q_annual,
    "CAPM-implied P mu_CAPM (annual)": 12 * (P @ mu_capm),
    "view - CAPM (annual)": q_annual - 12 * (P @ mu_capm),
    "std of view (annual)": 12 * np.sqrt(np.diag(Omega)),
})
display(cmp.round(4))

In [ ]:
diff = q_annual - 12 * (P @ mu_capm)
display(Markdown(
    "**Views relative to the CAPM.** The table compares each view with what the CAPM already implies for the same portfolio: "
    + "; ".join(f"{v}: the CAPM implies {12 * (P @ mu_capm)[v]:.1%}, so the view is "
                f"{'more optimistic' if diff[v] > 0 else 'more pessimistic'} by {abs(diff[v]):.1%}"
                for v in views)
    + ". Black–Litterman will move the expected returns away from the CAPM only in the direction of these differences."
))

## Q1.5

*Step 3: Use these views to compute the conditional expected excess returns ($\mu_{BL}$) and covariance matrix of excess returns ($\Sigma_{BL}$).*

### Answer

The posterior (Black–Litterman) moments are (Lecture 5, equations (10)–(11)):
$$
\mu_{BL} = \big[(\tau\Sigma)^{-1} + P^\top\Omega^{-1}P\big]^{-1}\big[(\tau\Sigma)^{-1}\mu_{CAPM} + P^\top\Omega^{-1}q\big],
$$
$$
\Sigma_{BL} = \Sigma + \big[(\tau\Sigma)^{-1} + P^\top\Omega^{-1}P\big]^{-1}.
$$
$\mu_{BL}$ is a precision-weighted average of the CAPM-implied returns and the views. $\Sigma_{BL}$ adds to $\Sigma$ the remaining uncertainty about the mean. We also check $\mu_{BL}$ with the equivalent "regression" form, $\mu_{BL} = \mu_{CAPM} + \tau\Sigma P^\top (P\tau\Sigma P^\top + \Omega)^{-1}(q - P\mu_{CAPM})$.

In [ ]:
S = Sigma.values
tS_inv = np.linalg.inv(tau * S)
O_inv = np.linalg.inv(Omega.values)
Pm = P.values

M = tS_inv + Pm.T @ O_inv @ Pm                     # posterior precision of mu
M_inv = np.linalg.inv(M)

mu_bl = pd.Series(M_inv @ (tS_inv @ mu_capm.values + Pm.T @ O_inv @ q.values), index=tickers)
Sigma_bl = pd.DataFrame(S + M_inv, index=tickers, columns=tickers)

# Check with the "regression" form of the posterior mean
mu_bl_2 = mu_capm.values + tau * S @ Pm.T @ np.linalg.solve(Pm @ (tau * S) @ Pm.T + Omega.values,
                                                            q.values - Pm @ mu_capm.values)
print("Both formulas give the same mu_BL:", np.allclose(mu_bl, mu_bl_2))

display(pd.DataFrame({
    "mu_CAPM (annual)": 12 * mu_capm,
    "mu_BL (annual)": 12 * mu_bl,
    "change (annual)": 12 * (mu_bl - mu_capm),
    "sample mean (annual)": 12 * mu,
}).round(4))
print("Sigma_BL (monthly):")
display(Sigma_bl.round(5))
print(f"Sigma_BL / Sigma on the diagonal (= 1 + tau = {1 + tau:.4f} with no views):")
display((np.diag(Sigma_bl) / np.diag(S)).round(4))
print("Implied views after updating, P mu_BL (annual):")
display((12 * (P @ mu_bl)).round(4))

In [ ]:
chg = mu_bl - mu_capm
display(Markdown(
    "**Result.** Annualised, the Black–Litterman expected excess returns are "
    + ", ".join(f"{t} {12 * mu_bl[t]:.1%}" for t in tickers)
    + ". Compared with the CAPM: "
    + ", ".join(f"{t} {12 * chg[t]:+.1%}" for t in tickers)
    + ". The views pull each expected return part of the way from the CAPM value towards the view "
    "(a precision-weighted average), not all the way. "
    f"AMZN has no view, but its expected return still changes ({12 * chg['AMZN']:+.1%} per year), "
    "because AMZN is correlated with the stocks in the views and the views carry information about it. "
    "The posterior covariance $\\Sigma_{BL}$ is slightly larger than $\\Sigma$ "
    f"(the diagonal is between {np.min(np.diag(Sigma_bl) / np.diag(S)):.4f} and {np.max(np.diag(Sigma_bl) / np.diag(S)):.4f} times the sample variances): "
    "it adds the uncertainty about the mean, which the views reduce relative to the no-view case $(1+\\tau)\\Sigma$."
))

## Q1.6

*Step 4: Now, use $\mu_{BL}$ and $\Sigma_{BL}$ to compute the Black-Litterman mean-variance weights.*

### Answer

We use the posterior moments in the Markowitz formula with the same risk aversion:
$$
w_{BL} = \frac{1}{\gamma_{mkt}}\,\Sigma_{BL}^{-1}\,\mu_{BL}.
$$
Following He and Litterman (1999), these weights can also be written as $w_{BL} = \frac{1}{1+\tau}\big(w_{mkt} + P^\top\Lambda\big)$. Without views, the investor holds $w_{mkt}/(1+\tau)$. The views tilt this portfolio **only for the assets that appear in the views**, in proportion to the rows of $P$.

In [ ]:
w_bl = pd.Series(np.linalg.solve(Sigma_bl, mu_bl) / gamma_mkt, index=tickers)
w_noview = w_mkt / (1 + tau)
tilt = w_bl - w_noview

# Lambda from the He-Litterman decomposition: (1+tau) * tilt = P' Lambda
Lam = pd.Series(np.linalg.lstsq(Pm.T, (1 + tau) * tilt.values, rcond=None)[0], index=views)

display(pd.DataFrame({"w_BL": w_bl, "w_mkt/(1+tau)": w_noview, "tilt": tilt, "w_mkt": w_mkt}).round(4))
print(f"Sum of BL weights: {w_bl.sum():.4f}   ->   risk-free weight: {1 - w_bl.sum():.4f}")
print(f"Tilt on AMZN (no view): {tilt['AMZN']:.2e}")
print("Lambda (one per view):"); display(Lam.round(4))
print("Check (1+tau)*tilt = P' Lambda:", np.allclose((1 + tau) * tilt.values, Pm.T @ Lam.values))

In [ ]:
display(Markdown(
    "**Result.** The Black–Litterman weights are "
    + ", ".join(f"{t} {w_bl[t]:.1%}" for t in tickers)
    + f", summing to {w_bl.sum():.3f} (the rest, {1 - w_bl.sum():.3f}, is in the risk-free asset). "
    "Relative to the no-view portfolio $w_{mkt}/(1+\\tau)$, the tilts are "
    + ", ".join(f"{t} {tilt[t]:+.1%}" for t in tickers)
    + ". As predicted by He and Litterman (1999), **AMZN's weight does not change** "
    f"(tilt {tilt['AMZN']:.1e}, i.e. zero) even though its expected return changed in Q1.5: "
    "the optimiser already accounts for its correlation with the other stocks through $\\Sigma$. "
    + " ".join(f"The view on {t} is {'below' if diff.iloc[i] < 0 else 'above'} the CAPM-implied return and its weight "
               f"{'falls' if tilt[t] < 0 else 'rises'} ({tilt[t]:+.1%})." for i, t in enumerate(["AAPL", "MSFT"]))
    + f" TSLA appears in both relative views, so its tilt is $-(\\Lambda_3+\\Lambda_4)/(1+\\tau)$ = {tilt['TSLA']:+.1%}, "
    f"while NVDA receives $\\Lambda_3/(1+\\tau)$ = {tilt['NVDA']:+.1%} and META $\\Lambda_4/(1+\\tau)$ = {tilt['META']:+.1%}. "
    + (f"Overall, the BL weights sum to less than one: the views are on average less optimistic than the CAPM, "
       f"so the investor holds less in the risky assets and {1 - w_bl.sum():.1%} in the risk-free asset."
       if w_bl.sum() < 1 else
       f"Overall, the BL weights sum to more than one, so the investor borrows {w_bl.sum() - 1:.1%} at the risk-free rate.")
))

## Q1.7

*Finally, compare the three sets of portfolio weights you have computed:*
- *Markowitz mean-variance portfolio weights;*
- *Capitalization-weighted (market) portfolio weights;*
- *Black-Litterman mean-variance portfolio weights.*

*Which weights seem the most reasonable? Why?*

In [ ]:
W = pd.DataFrame({"Markowitz": w_mv, "Market (cap-weighted)": w_mkt, "Black-Litterman": w_bl})

def describe(w):
    return pd.Series({
        "sum of risky weights": w.sum(),
        "risk-free weight": 1 - w.sum(),
        "gross exposure sum|w|": w.abs().sum(),
        "number of short positions": int((w < 0).sum()),
        "largest |weight|": w.abs().max(),
        "distance from market sum|w - w_mkt|": (w - w_mkt).abs().sum(),
    })

summary = W.apply(describe)
display(W.round(4))
display(summary.round(4))

ax = W.plot.bar(figsize=(9, 4), rot=0)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("portfolio weight")
ax.set_title("Markowitz vs market vs Black-Litterman weights")
plt.tight_layout()
plt.show()

In [ ]:
# Ex-ante annual Sharpe ratios of the three portfolios under different beliefs about the means
def sharpe(w, m, V):
    return np.sqrt(12) * (w @ m) / np.sqrt(w @ V @ w)

sr = pd.DataFrame({name: {"using sample means": sharpe(w, mu, Sigma),
                          "using CAPM means": sharpe(w, mu_capm, Sigma),
                          "using BL means": sharpe(w, mu_bl, Sigma_bl)}
                   for name, w in W.items()})
display(sr.round(3))

In [ ]:
display(Markdown(
    f"**Comparison.** The Markowitz portfolio has a gross exposure of {w_mv.abs().sum():.2f}, "
    f"{int((w_mv < 0).sum())} short position(s) and a largest absolute weight of {w_mv.abs().max():.1%}. "
    f"Its distance from the market portfolio is {(w_mv - w_mkt).abs().sum():.2f}. "
    f"The Black–Litterman portfolio has a gross exposure of {w_bl.abs().sum():.2f}, "
    f"{int((w_bl < 0).sum())} short position(s), a largest weight of {w_bl.abs().max():.1%}, "
    f"and a distance from the market of only {(w_bl - w_mkt).abs().sum():.2f}. "
    f"The Markowitz portfolio has the highest Sharpe ratio when evaluated with the sample means "
    f"({sr.loc['using sample means', 'Markowitz']:.2f}), but this is true by construction: "
    "it is optimised on those same means. Evaluated with the CAPM or BL means, its Sharpe ratio "
    f"falls to {sr.loc['using CAPM means', 'Markowitz']:.2f} and {sr.loc['using BL means', 'Markowitz']:.2f}."
))

### Answer: which weights are the most reasonable?

**The Black–Litterman weights are the most reasonable.**

- **Markowitz** weights are the least reasonable. They are extreme (large long and short positions and a large total exposure), they come from 96 noisy monthly means, and they would change a lot with a slightly different sample. Their high in-sample Sharpe ratio is an illusion created by optimising on the same data: as shown in Lectures 2–4, such portfolios perform poorly out of sample.
- **Market (capitalization-weighted)** weights are sensible: they are positive, diversified, require no estimation of means, and are the CAPM benchmark. But they ignore any information the investor has, here the views on AAPL, MSFT, NVDA, TSLA and META.
- **Black–Litterman** weights combine the strengths of both. They start from the market portfolio, which is a sensible anchor, and move away from it **only** for the assets in the views, **only** in the direction in which the views differ from the CAPM, and **only** in proportion to the confidence in the views ($\Omega$). The result is much closer to the market than the Markowitz portfolio, and every tilt is intuitive:
  - AAPL and MSFT are under-weighted if their views are below the CAPM-implied returns (and over-weighted if above);
  - NVDA, TSLA and META move with the two relative views;
  - AMZN, which has no view, keeps its market weight (divided by $1+\tau$).

  Each deviation from the market can be explained by a view, which also makes the portfolio easy to explain to a client.

In short, the Markowitz portfolio overfits the sample means, the market portfolio ignores the investor's views, and Black–Litterman uses the views while keeping the portfolio diversified and stable. This is why it is widely used in practice.